Maintenant, nous allons voir les models d'ia pour la prediciton des retards
Nous allons utilisé skilit learn

In [15]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df = pd.read_csv("cleaned_dataset.csv", sep=",")

# Column sauvegardé
categorical_cols = ["Service", "Departure station", "Arrival station"]

# Nettoyage (optionel si df nettoyé)
for col in df.columns:
    if col in categorical_cols:
        continue
    if pd.api.types.is_string_dtype(df[col]):
        cleaned = (
            df[col]
            .astype(str)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False)
            .str.replace(r"\s*min$", "", regex=True)
            .str.replace(r"[^0-9.\-]+", "", regex=True)
        )
        df[col] = pd.to_numeric(cleaned, errors="coerce")

# Cible
target = "Average delay of all trains at arrival"

drop_cols = [
    "Date", "Cancellation comments", "Departure delay comments",
    "Arrival delay comments", "Number of trains delayed > 15min",
    "Average delay of trains > 15min (if competing with flights)",
    "Number of trains delayed > 30min", "Number of trains delayed > 60min",
    target
]

X = df.drop(columns=drop_cols)
y = pd.to_numeric(df[target], errors="coerce")

X = pd.get_dummies(X, columns=categorical_cols)

non_numeric_cols = X.select_dtypes(exclude=["number"]).columns
if len(non_numeric_cols) > 0:
    X = X.drop(columns=non_numeric_cols)

# Supprimer les colonnes 100% manquantes
all_nan_cols = [col for col in X.columns if X[col].notna().sum() == 0]
if all_nan_cols:
    X = X.drop(columns=all_nan_cols)

# Supprimer les lignes avec valeurs manquantes
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

Notre IA doit donc s'entrainer et tester. Il lui definie donc des paramètres pour savoir sur combien de pourcents du csv il s'entrainera. Ici on lui definie 0.2 correspondant a 20 % du csv

In [16]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


On definie donc le model de l'ia et son entrainement ici un model Linear Regression


In [17]:
# Modèle
model = LinearRegression()
model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

MAE : 1.67
MSE  : 10.35
R²   : 0.4397


On obtient donc 3 resultats du model 

MAE : mean absolute error il se trompe donc de 1,64 min en moyenne
MSE : mean squared error, la plus haute erreur dans les pires cas
R² : de 0 a 1, determine si le model est pertinent au plus proche de 1, ici il l'est moyennement


On test un autre model, le randomForestRegressor

In [18]:
# Modèle
model = RandomForestRegressor()
model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

MAE : 1.25
MSE  : 8.03
R²   : 0.5653


On obtient de plus nette performance car les erreur de retard son tres peut lineaire d'ou les performances accrues du random forest

In [19]:
# Modèle
model = MLPRegressor()
model.fit(X_train, y_train)

# Evaluation

y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

MAE : 1.80
MSE  : 11.06
R²   : 0.4012


In [20]:
model = DecisionTreeRegressor()
model.fit(X_train, y_train)

# Evaluation

y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

MAE : 1.94
MSE  : 14.91
R²   : 0.1927


In [21]:
model = GradientBoostingRegressor()
model.fit(X_train, y_train)

# Evaluation

y_pred = model.predict(X_test)
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")


MAE : 1.35
MSE  : 8.12
R²   : 0.5601


In [22]:
import xgboost as xgb
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

df = pd.read_csv("cleaned_dataset.csv", sep=",", on_bad_lines="skip")

# Nettoyage
categorical_cols = ["Service", "Departure station", "Arrival station"]

for col in df.columns:
    if col in categorical_cols:
        continue
    if pd.api.types.is_string_dtype(df[col]):
        cleaned = (
            df[col].astype(str)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False)
            .str.replace(r"\s*min$", "", regex=True)
            .str.replace(r"[^0-9.\-]+", "", regex=True)
        )
        df[col] = pd.to_numeric(cleaned, errors="coerce")

# Cible
target = "Average delay of all trains at arrival"

drop_cols = [
    "Date", "Cancellation comments", "Departure delay comments",
    "Arrival delay comments", "Number of trains delayed > 15min",
    "Average delay of trains > 15min (if competing with flights)",
    "Number of trains delayed > 30min", "Number of trains delayed > 60min",
    target
]

X = df.drop(columns=drop_cols)
y = pd.to_numeric(df[target], errors="coerce")

X = pd.get_dummies(X, columns=categorical_cols)

non_numeric_cols = X.select_dtypes(exclude=["number"]).columns
if len(non_numeric_cols) > 0:
    X = X.drop(columns=non_numeric_cols)

mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modèle XGBoost
model = xgb.XGBRegressor(
    n_estimators=5000,    # nombre d'arbres
    max_depth=6,         # profondeur max de chaque arbre
    learning_rate=0.001,   # vitesse d'apprentissage
    random_state=42
)

# Entraînement
model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"MAE  : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

# Sauvegarde
joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

# Recharger et réutiliser
# model = joblib.load("model.pkl")
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# model.fit(X_train, y_train)



RMSE : 3.49
MAE  : 1.36
R²   : 0.3389
Modèle sauvegardé dans model.pkl


In [23]:
import xgboost as xgb
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

df = pd.read_csv("cleaned_dataset.csv", sep=",", on_bad_lines="skip")

# Nettoyage
categorical_cols = ["Service", "Departure station", "Arrival station"]

for col in df.columns:
    if col in categorical_cols:
        continue
    if pd.api.types.is_string_dtype(df[col]):
        cleaned = (
            df[col].astype(str)
            .str.replace(" ", "", regex=False)
            .str.replace(",", ".", regex=False)
            .str.replace(r"\s*min$", "", regex=True)
            .str.replace(r"[^0-9.\-]+", "", regex=True)
        )
        df[col] = pd.to_numeric(cleaned, errors="coerce")

# Cible
target = "Average delay of all trains at arrival"

drop_cols = [
    "Date", "Cancellation comments", "Departure delay comments",
    "Arrival delay comments", "Number of trains delayed > 15min",
    "Average delay of trains > 15min (if competing with flights)",
    "Number of trains delayed > 30min", "Number of trains delayed > 60min",
    target
]

X = df.drop(columns=drop_cols)
y = pd.to_numeric(df[target], errors="coerce")

X = pd.get_dummies(X, columns=categorical_cols)

non_numeric_cols = X.select_dtypes(exclude=["number"]).columns
if len(non_numeric_cols) > 0:
    X = X.drop(columns=non_numeric_cols)

mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

model = joblib.load("model.pkl")

# Continuer l'entraînement depuis l'ancien modèle
model.fit(
    X_train, y_train,
    xgb_model=model  # <-- c'est ça qui continue au lieu de repartir de zéro
)
# Évaluation
y_pred = model.predict(X_test)
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"MAE  : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

# Sauvegarde du modèle amélioré
joblib.dump(model, "model.pkl")
print("Modèle mis à jour sauvegardé !")


RMSE : 2.49
MAE  : 1.14
R²   : 0.6525
Modèle mis à jour sauvegardé !


In [24]:
import joblib
import pandas as pd
import numpy as np

model = joblib.load("model.pkl")

# Voir les features que le modèle attend
print(model.feature_names_in_)

['Unnamed: 0' 'Average journey time' 'Number of scheduled trains'
 'Number of cancelled trains' 'Number of trains delayed at departure'
 'Average delay of late trains at departure'
 'Average delay of all trains at departure'
 'Number of trains delayed at arrival'
 'Average delay of late trains at arrival'
 'Pct delay due to external causes' 'Pct delay due to infrastructure'
 'Pct delay due to traffic management' 'Pct delay due to rolling stock'
 'Pct delay due to station management and equipment reuse'
 'Pct delay due to passenger handling (crowding, disabled persons, connections)']


Nous avons des modeles via sklearn. Maintenant nous allons utilisés xgboost qui permet de creer nous meme des ia avec des variables spécifiques.

In [25]:
import xgboost as xgb
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

df = pd.read_csv("cleaned_dataset.csv", sep=",", on_bad_lines="skip")

# Feature engineering temporel
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m")
df["month"] = df["Date"].dt.month
df["year"] = df["Date"].dt.year
df["day_of_week"] = df["Date"].dt.dayofweek
df["is_peak_month"] = df["month"].isin([7, 8, 12]).astype(int)
df = df.drop(columns=["Date"])

# Cible
target = "Average delay of all trains at arrival"
y = pd.to_numeric(df[target], errors="coerce")

# Feature retenue
features = [
    "Departure station",
    "Arrival station",
    "Service",
    "month",
    "year",
    "day_of_week",
    "is_peak_month",
]

X = df[features]

# Encoder les catégorielles
X = pd.get_dummies(X, columns=["Service", "Departure station", "Arrival station"])

# Supprimer les lignes incomplètes
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modèle
model = xgb.XGBRegressor(
    n_estimators=10000,
    max_depth=6,
    learning_rate=0.001,
    random_state=42
)

model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"MAE  : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

# Sauvegarde en un seul fichier
joblib.dump({"model": model, "columns": X.columns.tolist()}, "model.pkl")
print("Modèle sauvegardé !")

RMSE : 3.58
MAE  : 2.21
R²   : 0.3056
Modèle sauvegardé !


Nous allons donc utilisé le model IA pour predire les retard moyens sur une ligne spécifique

In [ ]:
import joblib
import pandas as pd

# Charger le tout en un seul coup
saved = joblib.load("model.pkl")
model = saved["model"]
model_columns = saved["columns"]

# Remplissage
data = {
    "month": 12,
    "year": 2028,
    "day_of_week": 0,
    "is_peak_month": 0,
    "Service_National": 1,
    "Departure station_Bordeaux Saint Jean": 1,
    "Arrival station_Paris Montparnasse": 1,
}

X_input = pd.DataFrame([data], columns=model_columns).fillna(0)

prediction = model.predict(X_input)
print(f"Retard prédit : {prediction[0]:.1f} minutes")

Retard prédit : 7.5 minutes
